# Phase 1 — Dataset Exploration & Audit

**Project:** X-Ray Fracture Detection AI  
**Phase:** 1 — Data Pipeline  
**Task:** 1.1 — Dataset Exploration & Audit  

---

## Objective

This notebook performs a complete, reproducible audit of both source datasets:
- **FracAtlas** — Multi-region fracture atlas
- **GRAZPEDWRI-DX** — Pediatric wrist X-ray dataset

**All statistics are computed from actual files — nothing is hardcoded.**

---

## Contents
1. Setup & Path Resolution
2. Utility Functions
3. FracAtlas Audit
4. GRAZPEDWRI-DX Audit
5. Combined Statistics
6. Annotation Quality Report
7. Visualisations
8. Final Recommendations
9. Save Machine-Readable Report

## 1. Setup & Path Resolution

In [ ]:
import sys
import json
import csv
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import yaml

# -------------------------------------------------------
# Make src/ importable regardless of how notebook is launched
# -------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    AI_MODULE_ROOT = NOTEBOOK_DIR.parent
else:
    AI_MODULE_ROOT = NOTEBOOK_DIR

if str(AI_MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_MODULE_ROOT))

from src.utils.logger import get_logger
from src.utils.file_utils import find_images, IMAGE_EXTENSIONS

logger = get_logger('exploration')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

print(f'Notebook dir   : {NOTEBOOK_DIR}')
print(f'AI module root : {AI_MODULE_ROOT}')

In [ ]:
# -------------------------------------------------------
# Smart project root detection
# Looks for the directory containing BOTH ai_module/ AND a dataset folder
# -------------------------------------------------------

def find_project_root(start: Path) -> Path:
    """
    Walk up the directory tree to find the project root.
    Project root is the directory that contains:
      - ai_module/
      - at least one dataset directory
    """
    dataset_markers = {'fracatlas', 'FracAtlas', 'GRAZPEDWRI-DX', 'grazpedwri-dx'}
    
    for parent in list(start.parents)[:8]:
        has_ai_module = (parent / 'ai_module').exists()
        children = {c.name for c in parent.iterdir() if c.is_dir()}
        has_dataset = bool(children & dataset_markers)
        if has_ai_module and has_dataset:
            return parent
    
    # Fallback
    return AI_MODULE_ROOT.parent


def resolve_dataset_path(raw: str, project_root: Path, ai_root: Path) -> Path:
    """
    Resolve a dataset path from config.
    Tries multiple base directories in order.
    """
    p = Path(raw)
    if p.is_absolute() and p.exists():
        return p
    
    candidates = [
        (ai_root / raw).resolve(),
        (project_root / raw).resolve(),
        (project_root / Path(raw).name).resolve(),
    ]
    for c in candidates:
        if c.exists():
            return c
    
    # Return first candidate even if not found — caller checks existence
    return candidates[0]


PROJECT_ROOT = find_project_root(AI_MODULE_ROOT)
print(f'Project root detected : {PROJECT_ROOT}')

# List top-level contents
print(f'\nProject root contents:')
for item in sorted(PROJECT_ROOT.iterdir()):
    tag = '[DIR] ' if item.is_dir() else '[FILE]'
    print(f'  {tag} {item.name}')

In [ ]:
# -------------------------------------------------------
# Load dataset config
# -------------------------------------------------------
CONFIG_PATH = AI_MODULE_ROOT / 'configs' / 'dataset.yaml'
print(f'Config path : {CONFIG_PATH}')
print(f'Exists      : {CONFIG_PATH.exists()}')

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        CONFIG = yaml.safe_load(f)
    print('\nConfig loaded successfully.')
else:
    CONFIG = {'sources': []}
    print('WARNING: Config not found — using defaults.')

# Parse source paths
SOURCES_CFG = {s['name']: s['local_path'] for s in CONFIG.get('sources', [])}
print(f'\nSources from config:')
for name, path in SOURCES_CFG.items():
    print(f'  {name}: {path}')

In [ ]:
# -------------------------------------------------------
# Resolve actual dataset paths
# -------------------------------------------------------
FRACATLAS_ROOT = resolve_dataset_path(
    SOURCES_CFG.get('FracAtlas', '../fracatlas'),
    PROJECT_ROOT,
    AI_MODULE_ROOT
)

GRAZPEDWRI_ROOT = resolve_dataset_path(
    SOURCES_CFG.get('GRAZPEDWRI-DX', '../GRAZPEDWRI-DX'),
    PROJECT_ROOT,
    AI_MODULE_ROOT
)

REPORTS_DIR = AI_MODULE_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Path Resolution Results')
print('=' * 60)
print(f'FracAtlas root     : {FRACATLAS_ROOT}')
print(f'FracAtlas exists   : {FRACATLAS_ROOT.exists()}')
print(f'GRAZPEDWRI root    : {GRAZPEDWRI_ROOT}')
print(f'GRAZPEDWRI exists  : {GRAZPEDWRI_ROOT.exists()}')
print(f'Reports dir        : {REPORTS_DIR}')
print('=' * 60)

if not FRACATLAS_ROOT.exists():
    print('\n⚠️  FracAtlas not found. Searched:')
    for base in [AI_MODULE_ROOT, PROJECT_ROOT]:
        print(f'   {(base / "../fracatlas").resolve()}')
    
if not GRAZPEDWRI_ROOT.exists():
    print('\n⚠️  GRAZPEDWRI-DX not found. Searched:')
    for base in [AI_MODULE_ROOT, PROJECT_ROOT]:
        print(f'   {(base / "../GRAZPEDWRI-DX").resolve()}')

## 2. Utility Functions

In [ ]:
def scan_image_stats(image_paths: List[Path]) -> pd.DataFrame:
    """
    Scan a list of images and return a DataFrame with statistics.
    Handles empty list gracefully.
    """
    if not image_paths:
        return pd.DataFrame(columns=[
            'path', 'width', 'height', 'channels',
            'format', 'size_bytes', 'status'
        ])
    
    rows = []
    for p in image_paths:
        try:
            size_bytes = p.stat().st_size
            
            if size_bytes == 0:
                rows.append({
                    'path': str(p), 'width': None, 'height': None,
                    'channels': None, 'format': p.suffix.lower(),
                    'size_bytes': 0, 'status': 'zero_byte'
                })
                continue
            
            if p.suffix.lower() == '.dcm':
                from src.utils.image_utils import load_image
                img = load_image(p)
            else:
                img = cv2.imread(str(p))
            
            if img is None:
                rows.append({
                    'path': str(p), 'width': None, 'height': None,
                    'channels': None, 'format': p.suffix.lower(),
                    'size_bytes': size_bytes, 'status': 'corrupted'
                })
            else:
                h, w = img.shape[:2]
                c = img.shape[2] if img.ndim == 3 else 1
                rows.append({
                    'path': str(p), 'width': w, 'height': h,
                    'channels': c, 'format': p.suffix.lower(),
                    'size_bytes': size_bytes, 'status': 'ok'
                })
        except Exception as e:
            rows.append({
                'path': str(p), 'width': None, 'height': None,
                'channels': None, 'format': p.suffix.lower(),
                'size_bytes': 0, 'status': f'error'
            })
    
    return pd.DataFrame(rows)


def read_yolo_labels(label_dir: Path) -> pd.DataFrame:
    """
    Read all YOLO .txt labels in a directory into a DataFrame.
    Returns empty DataFrame (with correct columns) if no labels found.
    """
    cols = ['stem', 'class_id', 'x_center', 'y_center',
            'width', 'height', 'num_fields', 'status', 'raw']
    
    if not label_dir.exists():
        return pd.DataFrame(columns=cols)
    
    rows = []
    label_files = sorted(label_dir.glob('*.txt'))
    
    if not label_files:
        return pd.DataFrame(columns=cols)
    
    for lp in label_files:
        try:
            lines = lp.read_text(encoding='utf-8').strip().splitlines()
            valid_lines = [l for l in lines if l.strip()]
            
            if not valid_lines:
                rows.append({
                    'stem': lp.stem, 'class_id': None,
                    'x_center': None, 'y_center': None,
                    'width': None, 'height': None,
                    'num_fields': 0, 'status': 'empty', 'raw': ''
                })
                continue
            
            for line in valid_lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    try:
                        rows.append({
                            'stem': lp.stem,
                            'class_id': int(parts[0]),
                            'x_center': float(parts[1]),
                            'y_center': float(parts[2]),
                            'width': float(parts[3]),
                            'height': float(parts[4]),
                            'num_fields': 5,
                            'status': 'valid',
                            'raw': line
                        })
                    except ValueError:
                        rows.append({
                            'stem': lp.stem, 'class_id': None,
                            'x_center': None, 'y_center': None,
                            'width': None, 'height': None,
                            'num_fields': 5, 'status': 'parse_error',
                            'raw': line
                        })
                else:
                    rows.append({
                        'stem': lp.stem, 'class_id': None,
                        'x_center': None, 'y_center': None,
                        'width': None, 'height': None,
                        'num_fields': len(parts), 'status': 'malformed',
                        'raw': line
                    })
        except Exception as e:
            rows.append({
                'stem': lp.stem, 'class_id': None,
                'x_center': None, 'y_center': None,
                'width': None, 'height': None,
                'num_fields': 0, 'status': f'read_error',
                'raw': ''
            })
    
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)


def parse_voc_xml(xml_path: Path) -> dict:
    """Parse a Pascal VOC XML annotation file."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        objects = []
        for obj in root.findall('object'):
            name = obj.findtext('name', '').strip()
            bndbox = obj.find('bndbox')
            if bndbox is not None:
                try:
                    objects.append({
                        'name': name,
                        'xmin': float(bndbox.findtext('xmin', '0')),
                        'ymin': float(bndbox.findtext('ymin', '0')),
                        'xmax': float(bndbox.findtext('xmax', '0')),
                        'ymax': float(bndbox.findtext('ymax', '0')),
                        'has_bndbox': True,
                    })
                except ValueError:
                    objects.append({'name': name, 'has_bndbox': True, 'parse_error': True})
            else:
                objects.append({'name': name, 'has_bndbox': False})
        
        size_el = root.find('size')
        size = {}
        if size_el is not None:
            try:
                size = {
                    'width': int(size_el.findtext('width', '0')),
                    'height': int(size_el.findtext('height', '0')),
                }
            except ValueError:
                pass
        
        return {
            'stem': xml_path.stem,
            'filename': root.findtext('filename', ''),
            'size': size,
            'objects': objects,
            'num_objects': len(objects),
            'status': 'ok',
        }
    except ET.ParseError as e:
        return {'stem': xml_path.stem, 'status': f'parse_error', 'objects': [], 'num_objects': 0}
    except Exception as e:
        return {'stem': xml_path.stem, 'status': f'error', 'objects': [], 'num_objects': 0}


def describe_numeric(series: pd.Series, label: str) -> None:
    """Print descriptive statistics for a numeric Series."""
    valid = series.dropna()
    if len(valid) == 0:
        print(f'  {label}: no data')
        return
    print(
        f'  {label}:\n'
        f'    min={valid.min():.1f}  max={valid.max():.1f}  '
        f'mean={valid.mean():.1f}  median={valid.median():.1f}  '
        f'std={valid.std():.1f}'
    )


def safe_value_counts(series: pd.Series) -> dict:
    """Return value_counts as dict, handling empty series."""
    if series.empty:
        return {}
    return series.value_counts().to_dict()


print('✅ Utility functions defined.')

---
## 3. FracAtlas Audit

In [ ]:
print('=' * 60)
print('FRACATLAS')
print('=' * 60)
print(f'Root   : {FRACATLAS_ROOT}')
print(f'Exists : {FRACATLAS_ROOT.exists()}')

FA_AVAILABLE = FRACATLAS_ROOT.exists()

if not FA_AVAILABLE:
    print('\n⚠️  FracAtlas not available. Skipping FracAtlas audit.')
    print('   Cells below will report N/A for FracAtlas statistics.')

In [ ]:
if FA_AVAILABLE:
    print('FracAtlas — Directory Structure')
    print('-' * 40)
    for p in sorted(FRACATLAS_ROOT.rglob('*')):
        rel = p.relative_to(FRACATLAS_ROOT)
        depth = len(rel.parts)
        if depth <= 3:
            tag = '[DIR] ' if p.is_dir() else '[FILE]'
            print('  ' * (depth - 1) + f'{tag} {p.name}')
else:
    print('FracAtlas not available — skipped.')

In [ ]:
fa_image_paths = []
FA_IMAGES_DIR = None

if FA_AVAILABLE:
    # Find images directory
    for candidate_name in ['images', 'Images', 'image', 'img']:
        candidate = FRACATLAS_ROOT / candidate_name
        if candidate.exists():
            FA_IMAGES_DIR = candidate
            break
    
    if FA_IMAGES_DIR:
        fa_image_paths = find_images(FA_IMAGES_DIR)
        print(f'FracAtlas images dir : {FA_IMAGES_DIR}')
        print(f'Total images found   : {len(fa_image_paths)}')
        fa_fmt = Counter(p.suffix.lower() for p in fa_image_paths)
        print(f'Format distribution  : {dict(fa_fmt)}')
    else:
        # Search recursively
        fa_image_paths = find_images(FRACATLAS_ROOT)
        print(f'Images found (recursive): {len(fa_image_paths)}')
        if fa_image_paths:
            FA_IMAGES_DIR = fa_image_paths[0].parent
            fa_fmt = Counter(p.suffix.lower() for p in fa_image_paths)
            print(f'Format distribution: {dict(fa_fmt)}')
        else:
            print('WARNING: No images found in FracAtlas directory.')
else:
    print('FracAtlas not available — skipped.')

print(f'\nfa_image_paths count: {len(fa_image_paths)}')

In [ ]:
fa_img_df = pd.DataFrame(columns=['path', 'width', 'height', 'channels', 'format', 'size_bytes', 'status'])
fa_valid = fa_img_df.copy()

if FA_AVAILABLE and fa_image_paths:
    fa_sample = fa_image_paths[:500] if len(fa_image_paths) > 500 else fa_image_paths
    print(f'Scanning {len(fa_sample)} images for statistics...')
    
    fa_img_df = scan_image_stats(fa_sample)
    fa_valid = fa_img_df[fa_img_df['status'] == 'ok'].copy()
    
    print(f'\nFracAtlas Image Statistics (sample of {len(fa_sample)})')
    print(f'  Total scanned    : {len(fa_img_df)}')
    print(f'  Valid images     : {len(fa_valid)}')
    print(f'  Corrupted        : {(fa_img_df["status"] != "ok").sum()}')
    
    status_counts = safe_value_counts(fa_img_df['status'])
    print(f'  Status breakdown : {status_counts}')
    
    if len(fa_valid) > 0:
        print()
        describe_numeric(fa_valid['width'], 'Width (px)')
        describe_numeric(fa_valid['height'], 'Height (px)')
        ar = fa_valid['width'] / fa_valid['height']
        describe_numeric(ar, 'Aspect Ratio')
        print(f'  Channel modes    : {safe_value_counts(fa_valid["channels"])}')
else:
    if not FA_AVAILABLE:
        print('FracAtlas not available — skipped.')
    else:
        print('No images found in FracAtlas — skipping image statistics.')

display(fa_img_df.head(5))

In [ ]:
FA_YOLO_DIR = FRACATLAS_ROOT / 'Annotations' / 'YOLO' if FA_AVAILABLE else Path('/nonexistent')
fa_yolo_df = pd.DataFrame()

print(f'FracAtlas YOLO dir : {FA_YOLO_DIR}')
print(f'Exists             : {FA_YOLO_DIR.exists()}')

if FA_AVAILABLE and FA_YOLO_DIR.exists():
    fa_yolo_df = read_yolo_labels(FA_YOLO_DIR)
    
    if not fa_yolo_df.empty:
        total_label_files = fa_yolo_df['stem'].nunique()
        valid_rows = fa_yolo_df[fa_yolo_df['status'] == 'valid']
        empty_rows = fa_yolo_df[fa_yolo_df['status'] == 'empty']
        bad_rows = fa_yolo_df[~fa_yolo_df['status'].isin(['valid', 'empty'])]
        
        print(f'\nFracAtlas YOLO Label Audit')
        print(f'  Total label files  : {total_label_files}')
        print(f'  Valid box entries  : {len(valid_rows)}')
        print(f'  Empty label files  : {empty_rows["stem"].nunique()}')
        print(f'  Malformed entries  : {len(bad_rows)}')
        
        if len(valid_rows) > 0:
            print(f'  Class IDs seen     : {safe_value_counts(valid_rows["class_id"])}')
            
            # Coordinate range validation
            out_of_range = valid_rows[
                (valid_rows['x_center'] < 0) | (valid_rows['x_center'] > 1) |
                (valid_rows['y_center'] < 0) | (valid_rows['y_center'] > 1) |
                (valid_rows['width'] <= 0) | (valid_rows['width'] > 1) |
                (valid_rows['height'] <= 0) | (valid_rows['height'] > 1)
            ]
            print(f'  Out-of-range boxes : {len(out_of_range)}')
            
            # Boxes per image
            boxes_per_img = valid_rows.groupby('stem').size()
            print(f'\n  Boxes per positive image:')
            describe_numeric(boxes_per_img, 'Boxes/image')
    else:
        print('No YOLO labels found or all empty.')
else:
    if not FA_AVAILABLE:
        print('FracAtlas not available — skipped.')
    else:
        print('FracAtlas YOLO directory not found.')

In [ ]:
fa_coco = {}
FA_COCO_PATH = FRACATLAS_ROOT / 'Annotations' / 'COCO JSON' / 'COCO_fracture_masks.json' \
    if FA_AVAILABLE else Path('/nonexistent')

print(f'FracAtlas COCO path : {FA_COCO_PATH}')
print(f'Exists              : {FA_COCO_PATH.exists()}')

if FA_AVAILABLE and FA_COCO_PATH.exists():
    with open(FA_COCO_PATH, 'r', encoding='utf-8') as f:
        fa_coco = json.load(f)
    
    images_list = fa_coco.get('images', [])
    anns_list = fa_coco.get('annotations', [])
    cats_list = fa_coco.get('categories', [])
    
    print(f'\nFracAtlas COCO Statistics')
    print(f'  Images           : {len(images_list)}')
    print(f'  Annotations      : {len(anns_list)}')
    print(f'  Categories       : {cats_list}')
    
    has_seg = sum(1 for a in anns_list if a.get('segmentation'))
    has_bbox = sum(1 for a in anns_list if a.get('bbox'))
    print(f'  With segmentation: {has_seg}')
    print(f'  With bbox        : {has_bbox}')
    
    if anns_list:
        print(f'\n  Sample annotation keys: {list(anns_list[0].keys())}')
        print(f'  Sample bbox           : {anns_list[0].get("bbox")}')
else:
    if not FA_AVAILABLE:
        print('FracAtlas not available — skipped.')
    else:
        print('COCO annotation file not found.')

In [ ]:
fa_csv_df = pd.DataFrame()
FA_CSV = FRACATLAS_ROOT / 'dataset.csv' if FA_AVAILABLE else Path('/nonexistent')

print(f'FracAtlas dataset.csv : {FA_CSV}')
print(f'Exists                : {FA_CSV.exists()}')

if FA_AVAILABLE and FA_CSV.exists():
    fa_csv_df = pd.read_csv(FA_CSV)
    print(f'\nRows     : {len(fa_csv_df)}')
    print(f'Columns  : {list(fa_csv_df.columns)}')
    
    # Check for patient/study grouping fields
    grouping_candidates = [
        'patient_id', 'patientid', 'patient', 'study_id',
        'studyid', 'study', 'case_id', 'subject_id'
    ]
    found_group_fields = []
    for col in fa_csv_df.columns:
        if col.lower() in grouping_candidates:
            found_group_fields.append(col)
            print(f'  Potential grouping field: {col!r} — {fa_csv_df[col].nunique()} unique values')
    
    if not found_group_fields:
        print('  No patient/study grouping field detected in dataset.csv')
    
    print(f'\nSample rows:')
    display(fa_csv_df.head(5))
    
    print('\nCategorical columns:')
    for col in fa_csv_df.columns:
        if fa_csv_df[col].dtype == object and fa_csv_df[col].nunique() <= 20:
            print(f'  {col}: {safe_value_counts(fa_csv_df[col])}')
else:
    print('dataset.csv not found or FracAtlas not available.')

In [ ]:
fa_splits = {}
FA_SPLIT_DIR = FRACATLAS_ROOT / 'Utilities' / 'Fracture Split' if FA_AVAILABLE else Path('/nonexistent')

print(f'FracAtlas split dir : {FA_SPLIT_DIR}')
print(f'Exists              : {FA_SPLIT_DIR.exists()}')

if FA_AVAILABLE and FA_SPLIT_DIR.exists():
    for csv_name in ['train.csv', 'valid.csv', 'test.csv']:
        cp = FA_SPLIT_DIR / csv_name
        if cp.exists():
            df = pd.read_csv(cp)
            fa_splits[csv_name] = df
            print(f'\n  {csv_name}:')
            print(f'    Rows    : {len(df)}')
            print(f'    Columns : {list(df.columns)}')
            display(df.head(3))
    
    print(f'\nNOTE: We do NOT blindly use the FracAtlas original split.')
    print(f'      We will perform our own leakage-aware split for reproducibility.')
else:
    print('Split directory not found or FracAtlas not available.')

In [ ]:
fa_positive_count = 0
fa_negative_count = 0
fa_total_boxes = 0

if FA_AVAILABLE and not fa_yolo_df.empty:
    valid_rows = fa_yolo_df[fa_yolo_df['status'] == 'valid']
    empty_rows = fa_yolo_df[fa_yolo_df['status'] == 'empty']
    
    pos_stems = set(valid_rows['stem'].unique())
    neg_stems = set(empty_rows['stem'].unique())
    all_stems = fa_yolo_df['stem'].nunique()
    fa_total_boxes = len(valid_rows)
    fa_positive_count = len(pos_stems)
    fa_negative_count = len(neg_stems)
    
    print('FracAtlas — Positive/Negative Distribution')
    print('=' * 40)
    print(f'  Total label files         : {all_stems}')
    print(f'  Positive images (≥1 box)  : {fa_positive_count}')
    print(f'  Negative images (0 boxes) : {fa_negative_count}')
    print(f'  Total fracture boxes      : {fa_total_boxes}')
    
    if fa_positive_count > 0:
        boxes_per_img = valid_rows.groupby('stem').size()
        print(f'  Avg boxes/positive image  : {boxes_per_img.mean():.2f}')
        print(f'  Min boxes/positive image  : {boxes_per_img.min()}')
        print(f'  Max boxes/positive image  : {boxes_per_img.max()}')
    
    pos_ratio = fa_positive_count / all_stems if all_stems > 0 else 0
    print(f'\n  Positive ratio            : {pos_ratio:.2%}')
else:
    print('FracAtlas YOLO data not available — cannot compute positive/negative distribution.')

---
## 4. GRAZPEDWRI-DX Audit

In [ ]:
print('=' * 60)
print('GRAZPEDWRI-DX')
print('=' * 60)
print(f'Root   : {GRAZPEDWRI_ROOT}')
print(f'Exists : {GRAZPEDWRI_ROOT.exists()}')

GRZ_AVAILABLE = GRAZPEDWRI_ROOT.exists()

if not GRZ_AVAILABLE:
    print('\n⚠️  GRAZPEDWRI-DX not available. Skipping GRAZPEDWRI-DX audit.')

In [ ]:
if GRZ_AVAILABLE:
    print('GRAZPEDWRI-DX — Top-level structure')
    print('-' * 40)
    for item in sorted(GRAZPEDWRI_ROOT.iterdir()):
        tag = '[DIR] ' if item.is_dir() else '[FILE]'
        size_info = ''
        if item.is_dir():
            try:
                n = sum(1 for _ in item.rglob('*') if _.is_file())
                size_info = f'  ({n} files)'
            except:
                pass
        print(f'  {tag} {item.name}{size_info}')
else:
    print('GRAZPEDWRI-DX not available — skipped.')

In [ ]:
grz_image_paths = []
grz_image_parts = {}

if GRZ_AVAILABLE:
    print('GRAZPEDWRI-DX — Image Parts')
    print('-' * 40)
    
    for child in sorted(GRAZPEDWRI_ROOT.iterdir()):
        if child.is_dir() and child.name.lower().startswith('images'):
            part_images = find_images(child)
            grz_image_parts[child.name] = part_images
            grz_image_paths.extend(part_images)
            print(f'  {child.name}: {len(part_images)} images')
    
    if not grz_image_paths:
        # Try recursive search
        grz_image_paths = find_images(GRAZPEDWRI_ROOT)
        print(f'  (recursive search): {len(grz_image_paths)} images')
    
    print(f'\nTotal images : {len(grz_image_paths)}')
    grz_fmt = Counter(p.suffix.lower() for p in grz_image_paths)
    print(f'Formats      : {dict(grz_fmt)}')
else:
    print('GRAZPEDWRI-DX not available — skipped.')

print(f'\ngrz_image_paths count: {len(grz_image_paths)}')

In [ ]:
grz_img_df = pd.DataFrame(columns=['path', 'width', 'height', 'channels', 'format', 'size_bytes', 'status'])
grz_valid = grz_img_df.copy()

if GRZ_AVAILABLE and grz_image_paths:
    grz_sample = grz_image_paths[:500] if len(grz_image_paths) > 500 else grz_image_paths
    print(f'Scanning {len(grz_sample)} of {len(grz_image_paths)} images...')
    
    grz_img_df = scan_image_stats(grz_sample)
    grz_valid = grz_img_df[grz_img_df['status'] == 'ok'].copy()
    
    print(f'\nGRAZPEDWRI-DX Image Statistics (sample of {len(grz_sample)})')
    print(f'  Total scanned    : {len(grz_img_df)}')
    print(f'  Valid images     : {len(grz_valid)}')
    print(f'  Corrupted        : {(grz_img_df["status"] != "ok").sum()}')
    print(f'  Status breakdown : {safe_value_counts(grz_img_df["status"])}')
    
    if len(grz_valid) > 0:
        print()
        describe_numeric(grz_valid['width'], 'Width (px)')
        describe_numeric(grz_valid['height'], 'Height (px)')
        ar = grz_valid['width'] / grz_valid['height']
        describe_numeric(ar, 'Aspect Ratio')
        print(f'  Channel modes    : {safe_value_counts(grz_valid["channels"])}')
else:
    if not GRZ_AVAILABLE:
        print('GRAZPEDWRI-DX not available — skipped.')
    else:
        print('No images found in GRAZPEDWRI-DX.')

display(grz_img_df.head(5))

In [ ]:
GRZ_VOC_DIR = GRAZPEDWRI_ROOT / 'pascalvoc' if GRZ_AVAILABLE else Path('/nonexistent')
grz_voc_rows = []
grz_class_counter = Counter()

print(f'VOC dir  : {GRZ_VOC_DIR}')
print(f'Exists   : {GRZ_VOC_DIR.exists()}')

if GRZ_AVAILABLE and GRZ_VOC_DIR.exists():
    xml_files = list(GRZ_VOC_DIR.glob('*.xml'))
    print(f'XML files: {len(xml_files)}')
    
    parse_errors = 0
    
    for xml_path in xml_files:
        info = parse_voc_xml(xml_path)
        
        if info['status'] != 'ok':
            parse_errors += 1
            continue
        
        if not info['objects']:
            grz_voc_rows.append({
                'stem': info['stem'],
                'class_name': None,
                'has_bndbox': False,
                'status': 'no_objects',
            })
            continue
        
        for obj in info['objects']:
            class_name = obj.get('name', '')
            grz_class_counter[class_name] += 1
            grz_voc_rows.append({
                'stem': info['stem'],
                'class_name': class_name,
                'has_bndbox': obj.get('has_bndbox', False),
                'status': 'ok',
            })
    
    grz_voc_df = pd.DataFrame(grz_voc_rows)
    
    print(f'\nGRAZPEDWRI-DX Pascal VOC Annotation Audit')
    print(f'  Parse errors     : {parse_errors}')
    print(f'  Total XML files  : {len(xml_files)}')
    print(f'\n  Class distribution (ALL objects — NOT all are fracture):')
    
    FRACTURE_NAMES = {'fracture', 'Fracture', 'FRACTURE', 'bone fracture'}
    IGNORED_NAMES = {'text', 'Text', 'TEXT', 'ruler', 'artifact'}
    
    for cls_name, count in grz_class_counter.most_common():
        if cls_name in FRACTURE_NAMES:
            tag = '✅ → class 0 (fracture)'
        elif cls_name in IGNORED_NAMES:
            tag = '🚫 → will be IGNORED'
        elif cls_name is None or cls_name == '':
            tag = '⚠️  → empty class name'
        else:
            tag = '❓ → UNKNOWN (will be reported, not converted)'
        print(f'  {str(cls_name)!r:30s}: {count:6d}  {tag}')
    
    print(f'\n⚠️  CRITICAL: "text" objects are NOT fractures and will NOT be converted.')
else:
    grz_voc_df = pd.DataFrame()
    if not GRZ_AVAILABLE:
        print('GRAZPEDWRI-DX not available — skipped.')
    else:
        print('Pascal VOC directory not found.')

In [ ]:
GRZ_SUPERV_DIR = GRAZPEDWRI_ROOT / 'supervisely' / 'wrist' / 'ann' \
    if GRZ_AVAILABLE else Path('/nonexistent')
grz_superv_classes = Counter()

print(f'Supervisely dir : {GRZ_SUPERV_DIR}')
print(f'Exists          : {GRZ_SUPERV_DIR.exists()}')

if GRZ_AVAILABLE and GRZ_SUPERV_DIR.exists():
    superv_files = list(GRZ_SUPERV_DIR.glob('*.json'))
    print(f'Files           : {len(superv_files)}')
    
    if superv_files:
        # Inspect schema from first file
        with open(superv_files[0], 'r', encoding='utf-8') as f:
            superv_sample_data = json.load(f)
        
        print(f'\nSupervisely schema (first file):')
        print(f'  Top-level keys : {list(superv_sample_data.keys())}')
        
        objects = superv_sample_data.get('objects', [])
        print(f'  Objects count  : {len(objects)}')
        if objects:
            print(f'  Object keys    : {list(objects[0].keys())}')
            print(f'  Sample object  : {json.dumps(objects[0], indent=4)}')
        
        # Sample class distribution from up to 300 files
        sample_superv = superv_files[:300]
        for sp in sample_superv:
            try:
                with open(sp, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                for obj in data.get('objects', []):
                    cls = obj.get('classTitle', obj.get('class', 'unknown'))
                    grz_superv_classes[cls] += 1
            except:
                pass
        
        print(f'\nSupervisely class distribution (sample {len(sample_superv)} files):')
        for cls_name, count in grz_superv_classes.most_common():
            print(f'  {cls_name!r:30s}: {count}')
        
        print(f'\nNOTE: Supervisely annotations inspected for AUDIT only.')
        print(f'      Pascal VOC is selected as the canonical training annotation source.')
else:
    if not GRZ_AVAILABLE:
        print('GRAZPEDWRI-DX not available — skipped.')
    else:
        print('Supervisely annotations not found.')

In [ ]:
GRZ_CSV = GRAZPEDWRI_ROOT / 'dataset.csv' if GRZ_AVAILABLE else Path('/nonexistent')
grz_csv_df = pd.DataFrame()
grz_patient_field = None

print(f'GRAZPEDWRI-DX dataset.csv : {GRZ_CSV}')
print(f'Exists                    : {GRZ_CSV.exists()}')

if GRZ_AVAILABLE and GRZ_CSV.exists():
    grz_csv_df = pd.read_csv(GRZ_CSV)
    print(f'\nRows     : {len(grz_csv_df)}')
    print(f'Columns  : {list(grz_csv_df.columns)}')
    display(grz_csv_df.head(5))
    
    # Detect patient/study grouping field
    grouping_candidates = [
        'patient_id', 'patientid', 'patient', 'study_id',
        'studyid', 'study', 'case_id', 'id', 'subject_id'
    ]
    print('\nGrouping field analysis:')
    for col in grz_csv_df.columns:
        if col.lower() in grouping_candidates:
            unique_vals = grz_csv_df[col].nunique()
            total = len(grz_csv_df)
            coverage = grz_csv_df[col].notna().mean()
            print(
                f'  {col!r:25s}: {unique_vals} unique values, '
                f'{coverage:.1%} coverage → '
                f'{"USABLE" if coverage >= 0.8 else "INSUFFICIENT COVERAGE"}'
            )
            if coverage >= 0.8 and grz_patient_field is None:
                grz_patient_field = col
    
    if not grz_patient_field:
        print('  No reliable patient/study grouping field found.')
        print('  → Will use stratified image-level split.')
    else:
        print(f'\n  Selected grouping field: {grz_patient_field!r}')
else:
    print('dataset.csv not found or GRAZPEDWRI-DX not available.')

print(f'\nFinal patient/study field: {grz_patient_field}')

In [ ]:
grz_positive_count = 0
grz_negative_count = 0
grz_total_boxes = 0

FRACTURE_CLASS_NAMES_SET = {'fracture', 'Fracture', 'FRACTURE', 'bone fracture', 'Bone Fracture'}

if GRZ_AVAILABLE and grz_voc_rows:
    fracture_rows = [r for r in grz_voc_rows if r.get('class_name', '') in FRACTURE_CLASS_NAMES_SET]
    all_voc_stems = set(r['stem'] for r in grz_voc_rows)
    fracture_stems = set(r['stem'] for r in fracture_rows)
    no_fracture_stems = all_voc_stems - fracture_stems
    
    grz_positive_count = len(fracture_stems)
    grz_negative_count = len(no_fracture_stems)
    grz_total_boxes = len(fracture_rows)
    
    print('GRAZPEDWRI-DX — Positive/Negative Distribution')
    print('=' * 40)
    print(f'  Total annotated images   : {len(all_voc_stems)}')
    print(f'  Positive (fracture ≥1)   : {grz_positive_count}')
    print(f'  Without fracture obj     : {grz_negative_count}')
    print(f'  Total fracture boxes     : {grz_total_boxes}')
    
    if grz_positive_count > 0:
        print(f'  Avg boxes/positive image : {grz_total_boxes / grz_positive_count:.2f}')
    
    pos_ratio = grz_positive_count / len(all_voc_stems) if all_voc_stems else 0
    print(f'\n  Positive ratio           : {pos_ratio:.2%}')
else:
    print('GRAZPEDWRI-DX VOC data not available.')

---
## 5. Combined Statistics

In [ ]:
print('=' * 60)
print('COMBINED DATASET SUMMARY')
print('=' * 60)

fa_total = len(fa_image_paths)
grz_total = len(grz_image_paths)
combined_total = fa_total + grz_total

fa_pos = fa_positive_count
grz_pos = grz_positive_count
combined_pos = fa_pos + grz_pos
combined_neg = (fa_negative_count + grz_negative_count)
combined_boxes = fa_total_boxes + grz_total_boxes

print(f'\n  Dataset              Images    Positive    Negative    Boxes')
print(f'  {"─" * 60}')
print(f'  {"FracAtlas":20s} {fa_total:8d}  {fa_positive_count:10d}  {fa_negative_count:10d}  {fa_total_boxes:8d}')
print(f'  {"GRAZPEDWRI-DX":20s} {grz_total:8d}  {grz_positive_count:10d}  {grz_negative_count:10d}  {grz_total_boxes:8d}')
print(f'  {"─" * 60}')
print(f'  {"COMBINED":20s} {combined_total:8d}  {combined_pos:10d}  {combined_neg:10d}  {combined_boxes:8d}')

# Projected split sizes
if combined_total > 0:
    train_proj = int(combined_total * 0.70)
    val_proj   = int(combined_total * 0.15)
    test_proj  = combined_total - train_proj - val_proj
    
    print(f'\n  Projected splits (70/15/15):')
    print(f'  Train : ~{train_proj} images ({train_proj/combined_total:.1%})')
    print(f'  Val   : ~{val_proj} images ({val_proj/combined_total:.1%})')
    print(f'  Test  : ~{test_proj} images ({test_proj/combined_total:.1%})')
    
    overall_pos_ratio = combined_pos / combined_total
    print(f'\n  Overall positive ratio: {overall_pos_ratio:.2%}')
else:
    print('\n  No images found in either dataset.')

---
## 6. Annotation Quality Report

In [ ]:
print('ANNOTATION QUALITY REPORT')
print('=' * 60)

print('\n[FracAtlas — YOLO Annotations]')
if FA_AVAILABLE and not fa_yolo_df.empty:
    status_dist = safe_value_counts(fa_yolo_df['status'])
    for status, count in status_dist.items():
        print(f'  {status:20s}: {count}')
    
    valid_rows = fa_yolo_df[fa_yolo_df['status'] == 'valid']
    if not valid_rows.empty:
        issues = []
        for col in ['x_center', 'y_center']:
            out = valid_rows[(valid_rows[col] < 0) | (valid_rows[col] > 1)]
            if len(out) > 0:
                issues.append(f'{col} out of [0,1]: {len(out)}')
        for col in ['width', 'height']:
            out = valid_rows[(valid_rows[col] <= 0) | (valid_rows[col] > 1)]
            if len(out) > 0:
                issues.append(f'{col} invalid: {len(out)}')
        if issues:
            print(f'  ⚠️  Issues found:')
            for iss in issues:
                print(f'     {iss}')
        else:
            print('  ✅ No coordinate range issues found in valid entries.')
else:
    print('  Not available.')

print('\n[GRAZPEDWRI-DX — Pascal VOC Annotations]')
if GRZ_AVAILABLE and grz_class_counter:
    FRACTURE_NAMES = {'fracture', 'Fracture', 'FRACTURE', 'bone fracture'}
    IGNORED_NAMES = {'text', 'Text', 'TEXT', 'ruler', 'artifact'}
    
    for cls_name, count in grz_class_counter.most_common():
        if cls_name in FRACTURE_NAMES:
            tag = '✅ → class 0'
        elif cls_name in IGNORED_NAMES:
            tag = '🚫 → ignored'
        else:
            tag = '❓ → unknown'
        print(f'  {str(cls_name)!r:30s}: {count:6d}  {tag}')
else:
    print('  Not available.')

---
## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FracAtlas
ax = axes[0]
if len(fa_valid) > 0:
    ax.scatter(fa_valid['width'], fa_valid['height'], alpha=0.4, s=10, c='steelblue')
    ax.set_title(f'FracAtlas — Resolution Distribution\n(n={len(fa_valid)} sampled)')
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('FracAtlas — No Data')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')

# GRAZPEDWRI-DX
ax = axes[1]
if len(grz_valid) > 0:
    ax.scatter(grz_valid['width'], grz_valid['height'], alpha=0.4, s=10, c='coral')
    ax.set_title(f'GRAZPEDWRI-DX — Resolution Distribution\n(n={len(grz_valid)} sampled)')
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('GRAZPEDWRI-DX — No Data')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'resolution_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORTS_DIR / "resolution_distribution.png"}')

In [ ]:
# Positive vs Negative distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, pos, neg) in zip(axes, [
    ('FracAtlas', fa_positive_count, fa_negative_count),
    ('GRAZPEDWRI-DX', grz_positive_count, grz_negative_count),
]):
    if pos + neg > 0:
        ax.bar(['Positive\n(fracture)', 'Negative\n(no fracture)'],
               [pos, neg],
               color=['#e74c3c', '#2ecc71'],
               edgecolor='white', linewidth=1.5)
        ax.bar_label(ax.containers[0], fmt='%d')
        ax.set_title(f'{name}\nPositive/Negative Distribution')
        ax.set_ylabel('Image Count')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{name} — No Data')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'positive_negative_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORTS_DIR / "positive_negative_distribution.png"}')

In [ ]:
# GRAZPEDWRI-DX class distribution
if grz_class_counter:
    FRACTURE_NAMES = {'fracture', 'Fracture', 'FRACTURE', 'bone fracture'}
    IGNORED_NAMES = {'text', 'Text', 'TEXT', 'ruler', 'artifact'}
    
    classes = list(grz_class_counter.keys())
    counts = list(grz_class_counter.values())
    colors = [
        '#e74c3c' if c in FRACTURE_NAMES
        else '#95a5a6' if c in IGNORED_NAMES
        else '#f39c12'
        for c in classes
    ]
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(classes) * 0.6)))
    bars = ax.barh(classes, counts, color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%d', padding=3)
    ax.set_xlabel('Count')
    ax.set_title('GRAZPEDWRI-DX — Pascal VOC Class Distribution\n'
                 '(Red=fracture, Grey=ignored, Orange=unknown)')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'grazpedwri_class_distribution.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {REPORTS_DIR / "grazpedwri_class_distribution.png"}')
else:
    print('No class distribution data to plot.')

In [ ]:
# Sample FracAtlas images with bounding boxes
def show_samples_with_boxes(
    image_paths: List[Path],
    label_dir: Optional[Path],
    title: str,
    n: int = 4,
    save_name: Optional[str] = None
):
    """
    Display sample images with YOLO bounding boxes overlaid.
    Handles cases where images or labels are missing gracefully.
    """
    if not image_paths:
        print(f'{title}: No images to display.')
        return
    
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    
    shown = 0
    for img_path in image_paths:
        if shown >= n:
            break
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        boxes = []
        if label_dir is not None:
            lp = label_dir / f'{img_path.stem}.txt'
            if lp.exists():
                for line in lp.read_text().strip().splitlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        try:
                            boxes.append([float(x) for x in parts])
                        except ValueError:
                            pass
        
        ax = axes[shown]
        ax.imshow(img_rgb, cmap='gray' if img_rgb.ndim == 2 else None)
        ax.set_title(
            f'{img_path.name}\n{w}×{h} | boxes={len(boxes)}',
            fontsize=8
        )
        ax.axis('off')
        
        for box in boxes:
            _, xc, yc, bw, bh = box
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            rect = patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=2, edgecolor='#e74c3c', facecolor='none'
            )
            ax.add_patch(rect)
        
        shown += 1
    
    # Hide unused axes
    for i in range(shown, n):
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    if save_name:
        plt.savefig(REPORTS_DIR / save_name, dpi=100, bbox_inches='tight')
        print(f'Saved: {REPORTS_DIR / save_name}')
    plt.show()


# FracAtlas samples
if FA_AVAILABLE and fa_image_paths and FA_YOLO_DIR.exists():
    show_samples_with_boxes(
        fa_image_paths[:20],
        FA_YOLO_DIR,
        'FracAtlas — Sample Images with YOLO Bounding Boxes',
        n=4,
        save_name='samples_fracatlas.png'
    )
else:
    print('FracAtlas samples: not available.')

In [ ]:
# GRAZPEDWRI-DX samples — show from VOC converted on-the-fly
if GRZ_AVAILABLE and grz_image_paths and GRZ_VOC_DIR.exists():
    
    # Build a quick image stem → path map
    grz_img_map = {p.stem: p for p in grz_image_paths}
    
    # Find images that have fracture annotations
    FRACTURE_NAMES = {'fracture', 'Fracture', 'FRACTURE', 'bone fracture'}
    fracture_stems_grz = set(
        r['stem'] for r in grz_voc_rows
        if r.get('class_name', '') in FRACTURE_NAMES
    )
    sample_stems = list(fracture_stems_grz)[:8]
    sample_paths = [grz_img_map[s] for s in sample_stems if s in grz_img_map]
    
    if sample_paths:
        # For GRAZPEDWRI we show images, annotations come from VOC so draw manually
        n = min(4, len(sample_paths))
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        if n == 1:
            axes = [axes]
        
        for i, img_path in enumerate(sample_paths[:n]):
            img = cv2.imread(str(img_path))
            if img is None:
                axes[i].axis('off')
                continue
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            
            # Load VOC annotation
            xml_path = GRZ_VOC_DIR / f'{img_path.stem}.xml'
            boxes_to_draw = []
            if xml_path.exists():
                info = parse_voc_xml(xml_path)
                for obj in info.get('objects', []):
                    if obj.get('name', '') in FRACTURE_NAMES and 'xmin' in obj:
                        boxes_to_draw.append(obj)
            
            axes[i].imshow(img_rgb)
            axes[i].set_title(
                f'{img_path.name}\n{w}×{h} | fracture boxes={len(boxes_to_draw)}',
                fontsize=8
            )
            axes[i].axis('off')
            
            for obj in boxes_to_draw:
                x1, y1 = obj['xmin'], obj['ymin']
                bw = obj['xmax'] - obj['xmin']
                bh = obj['ymax'] - obj['ymin']
                rect = patches.Rectangle(
                    (x1, y1), bw, bh,
                    linewidth=2, edgecolor='#e74c3c', facecolor='none'
                )
                axes[i].add_patch(rect)
        
        plt.suptitle(
            'GRAZPEDWRI-DX — Sample Fracture Images with Pascal VOC Boxes',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        plt.savefig(REPORTS_DIR / 'samples_grazpedwri.png', dpi=100, bbox_inches='tight')
        print(f'Saved: {REPORTS_DIR / "samples_grazpedwri.png"}')
        plt.show()
    else:
        print('No fracture-annotated images found for visualisation.')
else:
    print('GRAZPEDWRI-DX samples: not available.')

---
## 8. Final Recommendations

In [ ]:
print('=' * 60)
print('FINAL RECOMMENDATIONS')
print('=' * 60)

print('''
CANONICAL ANNOTATION SOURCE:

  FracAtlas:
    Source  → YOLO (Annotations/YOLO/)
    Reason  → Already in YOLO format, validated as class 0 = fracture
    Fallback→ COCO JSON bbox if YOLO proves inconsistent
    Note    → COCO segmentation is retained in raw data for audit only

  GRAZPEDWRI-DX:
    Source  → Pascal VOC (pascalvoc/*.xml)
    Reason  → Most explicit and well-structured annotation format
    Note    → Supervisely inspected for audit only, NOT used as training source
    CRITICAL→ "text" objects MUST NOT become fracture class 0
    CRITICAL→ Only objects in FRACTURE_CLASS_NAMES → class 0

CLASS MAPPING:
    class 0 = fracture  (only class in Phase 1)
    "text", "ruler", "artifact" → reported and ignored
    Unknown class names → reported, NOT converted

SPLIT STRATEGY:
    1. Check for patient/study grouping in dataset.csv
    2. If reliable (≥80% coverage): group-level stratified split
    3. If not: image-level stratified split on fracture_positive
    Seed    : 42
    Target  : 70% train / 15% val / 15% test

EMPTY LABELS:
    → Valid negative samples (not errors)
    → Configurable via --strict flag

NEXT STEPS:
    1. python scripts/prepare_dataset.py --verbose
    2. python scripts/validate_dataset.py
    3. Check reports/validation_report.json
''')

print(f'Available datasets:')
print(f'  FracAtlas      : {"✅ Available" if FA_AVAILABLE else "❌ NOT FOUND"}')
print(f'  GRAZPEDWRI-DX  : {"✅ Available" if GRZ_AVAILABLE else "❌ NOT FOUND"}')

---
## 9. Save Machine-Readable Report

In [ ]:
exploration_report = {
    'phase': 1,
    'task': '1.1_data_exploration',
    'generated_by': 'notebooks/01_data_exploration.ipynb',
    'paths': {
        'project_root': str(PROJECT_ROOT),
        'ai_module_root': str(AI_MODULE_ROOT),
        'fracatlas_root': str(FRACATLAS_ROOT),
        'grazpedwri_root': str(GRAZPEDWRI_ROOT),
    },
    'availability': {
        'fracatlas': FA_AVAILABLE,
        'grazpedwri': GRZ_AVAILABLE,
    },
    'datasets': {
        'fracatlas': {
            'total_images': fa_total,
            'image_formats': dict(Counter(p.suffix.lower() for p in fa_image_paths)),
            'yolo_annotation_dir': str(FA_YOLO_DIR),
            'yolo_dir_exists': FA_YOLO_DIR.exists() if FA_AVAILABLE else False,
            'coco_annotation_exists': FA_COCO_PATH.exists() if FA_AVAILABLE else False,
            'positive_images': fa_positive_count,
            'negative_images': fa_negative_count,
            'total_boxes': fa_total_boxes,
            'canonical_annotation_source': 'YOLO_existing',
            'class_mapping': {'class_0': 'fracture'},
        },
        'grazpedwri': {
            'total_images': grz_total,
            'image_formats': dict(Counter(p.suffix.lower() for p in grz_image_paths)),
            'voc_annotation_dir': str(GRZ_VOC_DIR),
            'voc_dir_exists': GRZ_VOC_DIR.exists() if GRZ_AVAILABLE else False,
            'supervisely_dir_exists': GRZ_SUPERV_DIR.exists() if GRZ_AVAILABLE else False,
            'positive_images': grz_positive_count,
            'negative_images': grz_negative_count,
            'total_boxes': grz_total_boxes,
            'class_distribution': dict(grz_class_counter),
            'supervisely_class_distribution': dict(grz_superv_classes),
            'fracture_class_names': list(FRACTURE_CLASS_NAMES_SET),
            'ignored_class_names': ['text', 'Text', 'TEXT', 'ruler', 'artifact'],
            'patient_grouping_field': grz_patient_field,
            'canonical_annotation_source': 'Pascal_VOC',
        },
    },
    'combined': {
        'total_images': combined_total,
        'total_positive': combined_pos,
        'total_negative': combined_neg,
        'total_boxes': combined_boxes,
        'overall_positive_ratio': round(combined_pos / combined_total, 4) if combined_total > 0 else 0,
        'projected_train': int(combined_total * 0.70) if combined_total > 0 else 0,
        'projected_val': int(combined_total * 0.15) if combined_total > 0 else 0,
        'projected_test': combined_total - int(combined_total * 0.70) - int(combined_total * 0.15) if combined_total > 0 else 0,
    },
    'recommendations': {
        'fracatlas_annotation_source': 'YOLO_existing',
        'grazpedwri_annotation_source': 'Pascal_VOC',
        'split_strategy': 'group_level_if_available_else_stratified_image_level',
        'seed': 42,
        'nc': 1,
        'class_names': ['fracture'],
        'non_fracture_objects': 'reported_and_ignored',
        'empty_labels': 'valid_negative_samples',
    }
}

report_path = REPORTS_DIR / 'data_exploration.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(exploration_report, f, indent=2, ensure_ascii=False, default=str)

print(f'✅ Exploration report saved: {report_path}')
print(f'\nFinal Summary:')
print(f'  FracAtlas      : {fa_total} images | +{fa_positive_count} / -{fa_negative_count} | {fa_total_boxes} boxes')
print(f'  GRAZPEDWRI-DX  : {grz_total} images | +{grz_positive_count} / -{grz_negative_count} | {grz_total_boxes} boxes')
print(f'  Combined       : {combined_total} images | +{combined_pos} / -{combined_neg} | {combined_boxes} boxes')
print(f'\nNext: python scripts/prepare_dataset.py --verbose')